In [3]:

# ============================================================
# NLP + K-MEANS CLUSTERING
# Dataset: annual_csv.csv
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import warnings

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


# ============================================================
# 2. LOAD DATASET
# ============================================================

file_path = "annual_csv.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())


# ============================================================
# 3. BASIC DATASET INFORMATION
# ============================================================

print("\n========== DATASET INFORMATION ==========\n")

print("Shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# ============================================================
# 4. REMOVE EMPTY COLUMNS AND DUPLICATES
# ============================================================

# Remove completely empty columns
df = df.dropna(axis=1, how="all")

# Remove duplicate rows
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print("\nDataset after cleaning:")
print(df.shape)


# ============================================================
# 5. FIND TEXT COLUMNS
# ============================================================

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("\nText columns found:")
print(text_columns)


# ============================================================
# 6. CHECK THAT TEXT COLUMNS EXIST
# ============================================================

if len(text_columns) == 0:
    raise ValueError(
        "No text columns were found. "
        "NLP requires at least one text column."
    )


# ============================================================
# 7. COMBINE TEXT COLUMNS
# ============================================================

df["combined_text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

print("\nCombined text created.")

display(
    df[["combined_text"]].head()
)


# ============================================================
# 8. TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text):

    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove punctuation
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Remove leading/trailing spaces
    text = text.strip()

    return text


# ============================================================
# 9. APPLY TEXT CLEANING
# ============================================================

df["clean_text"] = df["combined_text"].apply(
    clean_text
)

print("\nCleaned text:")
display(
    df[["combined_text", "clean_text"]].head()
)


# ============================================================
# 10. REMOVE EMPTY TEXT ROWS
# ============================================================

df = df[
    df["clean_text"].str.strip() != ""
].copy()

df = df.reset_index(drop=True)

print("\nRows after removing empty text:")
print(len(df))


# ============================================================
# 11. TF-IDF VECTORIZATION
# ============================================================

print("\nCreating TF-IDF matrix...")

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(
    df["clean_text"]
)

print("TF-IDF matrix created.")
print("TF-IDF shape:", X.shape)


# ============================================================
# 12. GET TF-IDF FEATURES
# ============================================================

feature_names = tfidf.get_feature_names_out()

print("\nNumber of TF-IDF features:")
print(len(feature_names))

print("\nFirst 20 features:")
print(feature_names[:20])


# ============================================================
# 13. FIND OPTIMAL NUMBER OF CLUSTERS
# ============================================================

print("\nCalculating optimal number of clusters...")

# Maximum K should not exceed the number of documents
max_k = min(10, len(df) - 1)

if max_k < 2:
    raise ValueError(
        "There are not enough records to perform K-Means clustering."
    )

k_values = range(2, max_k + 1)

inertia_values = []
silhouette_values = []


for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    inertia_values.append(
        model.inertia_
    )

    silhouette_values.append(
        silhouette_score(X, labels)
    )


# ============================================================
# 14. ELBOW METHOD
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    inertia_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Choosing K")

plt.xticks(list(k_values))
plt.grid(True)

plt.show()


# ============================================================
# 15. SILHOUETTE SCORE
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    silhouette_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.xticks(list(k_values))
plt.grid(True)

plt.show()


# ============================================================
# 16. DISPLAY K AND SILHOUETTE SCORES
# ============================================================

score_table = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertia_values,
    "Silhouette_Score": silhouette_values
})

display(score_table)


# ============================================================
# 17. AUTOMATICALLY SELECT BEST K
# ============================================================

best_k = list(k_values)[
    np.argmax(silhouette_values)
]

print("\nBest K based on silhouette score:")
print(best_k)


# ============================================================
# 18. K-MEANS CLUSTERING
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

print("\nK-Means clustering completed!")


# ============================================================
# 19. CLUSTER COUNTS
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("\nNumber of records in each cluster:")

display(
    cluster_counts.to_frame(
        name="Number_of_Records"
    )
)


# ============================================================
# 20. TOP WORDS FOR EACH CLUSTER
# ============================================================

print("\n========== TOP WORDS IN EACH CLUSTER ==========\n")

centers = kmeans.cluster_centers_

for cluster_number in range(best_k):

    # Sort feature weights
    top_indices = centers[
        cluster_number
    ].argsort()[-20:][::-1]

    top_words = [
        feature_names[i]
        for i in top_indices
    ]

    print(
        f"Cluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )

    print("-" * 70)


# ============================================================
# 21. CREATE CLUSTER KEYWORDS TABLE
# ============================================================

cluster_keywords = []

for cluster_number in range(best_k):

    top_indices = centers[
        cluster_number
    ].argsort()[-20:][::-1]

    top_words = [
        feature_names[i]
        for i in top_indices
    ]

    cluster_keywords.append({
        "Cluster": cluster_number,
        "Top_Words": ", ".join(top_words)
    })


cluster_keywords_df = pd.DataFrame(
    cluster_keywords
)

print("\nCluster keyword summary:")

display(
    cluster_keywords_df
)


# ============================================================
# 22. REDUCE DIMENSIONS FOR VISUALIZATION
# ============================================================

print("\nReducing TF-IDF dimensions...")

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)

print(
    "Explained variance:",
    svd.explained_variance_ratio_.sum()
)


# ============================================================
# 23. VISUALIZE CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    alpha=0.7
)

plt.xlabel("SVD Component 1")
plt.ylabel("SVD Component 2")

plt.title(
    "K-Means Clustering Using NLP / TF-IDF"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(True)

plt.show()


# ============================================================
# 24. CALCULATE FINAL SILHOUETTE SCORE
# ============================================================

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

print(
    "\nFinal Silhouette Score:",
    round(final_silhouette, 4)
)


# ============================================================
# 25. DISPLAY RECORDS FROM EACH CLUSTER
# ============================================================

print("\n========== SAMPLE RECORDS BY CLUSTER ==========\n")

for cluster_number in range(best_k):

    print(
        f"\nCLUSTER {cluster_number}"
    )

    print("=" * 70)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    display(
        cluster_data.head(5)
    )


# ============================================================
# 26. CLUSTER SUMMARY
# ============================================================

cluster_summary = (
    df.groupby("cluster")
    .size()
    .reset_index(
        name="Number_of_Records"
    )
)

print("\nCluster summary:")

display(
    cluster_summary
)


# ============================================================
# 27. ADD CLUSTER NAMES
# ============================================================
#
# The names below are examples.
# CHANGE THEM according to the top words
# printed for your actual dataset.
#

cluster_names = {}

for cluster_number in range(best_k):

    cluster_names[
        cluster_number
    ] = f"Cluster {cluster_number}"


df["cluster_name"] = (
    df["cluster"]
    .map(cluster_names)
)


# ============================================================
# 28. FINAL DATASET PREVIEW
# ============================================================

print("\nFinal dataset:")

display(
    df.head()
)


# ============================================================
# 29. SAVE CLUSTERED DATASET
# ============================================================

output_file = (
    "annual_csv_clustered.csv"
)

df.to_csv(
    output_file,
    index=False
)

print(
    f"\nClustered dataset saved as: {output_file}"
)


# ============================================================
# 30. SAVE CLUSTER KEYWORDS
# ============================================================

keywords_file = (
    "cluster_keywords.csv"
)

cluster_keywords_df.to_csv(
    keywords_file,
    index=False
)

print(
    f"Cluster keywords saved as: {keywords_file}"
)


# ============================================================
# 31. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    f"Original/cleaned records: {len(df)}"
)

print(
    f"Number of clusters: {best_k}"
)

print(
    f"Number of TF-IDF features: {X.shape[1]}"
)

print(
    f"Final silhouette score: "
    f"{final_silhouette:.4f}"
)

print(
    "\nOutput files:"
)

print(
    "1. annual_csv_clustered.csv"
)

print(
    "2. cluster_keywords.csv"
)

print("\nProcessing completed successfully!")




Libraries imported successfully!
Dataset loaded successfully!
Number of rows: 70
Number of columns: 2


,Date,Price
0,1950-12,34.72
1,1951-12,34.66
2,1952-12,34.79
3,1953-12,34.85
4,1954-12,35.04



========== DATASET INFORMATION ==========

Shape:
(70, 2)

Column names:
['Date', 'Price']

Data types:
Date         str
Price    float64
dtype: object

Missing values:


Date     0
Price    0
dtype: int64


Duplicate rows:
0

Dataset after cleaning:
(70, 2)

Text columns found:
['Date']

Combined text created.


,combined_text
0,1950-12
1,1951-12
2,1952-12
3,1953-12
4,1954-12



Cleaned text:


,combined_text,clean_text
0,1950-12,
1,1951-12,
2,1952-12,
3,1953-12,
4,1954-12,



Rows after removing empty text:
0

Creating TF-IDF matrix...


ValueError: empty vocabulary; perhaps the documents only contain stop words